In [4]:
import os
import sys

# 取得目前執行 Notebook 的工作目錄
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)
# 強制重新載入 dbcon 模組（確保用到的是正確的 env）
# 刪除舊變數（避免殘留）
# import utils.dbcon
from common.utils.dbcon import engine

import importlib, resources.register, resources.login, resources.trails

# importlib.reload(utils.dbcon)
importlib.reload(resources.register)
importlib.reload(resources.login)
importlib.reload(resources.trails)
from pathlib import Path
import json
import pandas as pd

from flask_restful import Resource, reqparse
from app import app

# from utils.dbcon import engine

# from werkzeug.security import generate_password_hash, check_password_hash
# from sqlalchemy import text

from app import app  # 假設你的 Flask app 在 app.py

# 建立 Flask test client（不真正開 server）
client = app.test_client()

/home/zoe/allpass/backend


In [5]:
# Test: /api/register

new_user_email = "waxapple55@example.com"
new_user_password = "waxapple55"
new_user_username = "waxapple55"

payload = {
    "email": new_user_email,
    "password": new_user_password,
    "username": new_user_username,
}

response = client.post("/api/register", json=payload)  # 模擬前端送 JSON

print("Status:", response.status_code)
data = response.get_json()
print("JSON:", data)

assert response.status_code == 201
assert data["message"] == "User registered successfully"
assert data["data"]["username"] == new_user_username

Status: 201
JSON: {'message': 'User registered successfully', 'data': {'id': 529, 'username': 'waxapple55', 'created_at': '2025-08-25 17:17:40.162948+08:00'}}


In [6]:
# Test: /api/Login


payload = {"email": new_user_email, "password": new_user_password}

response = client.post("/api/login", json=payload)

print("Status", response.status_code)
# response.data 是 bytes 原始資料
print("Data", response.data)

# 把回傳的 JSON 轉成 Python dict 再存取
data = response.get_json()

assert data["data"]["username"] == new_user_username

Status 200
Data b'{\n    "message": "Login successful",\n    "data": {\n        "user_id": 529,\n        "username": "waxapple55"\n    }\n}\n'


In [14]:
# Test: /api/trails
response = client.get("/api/trails")

print("Status", response.status_code)
# response.data 是 bytes 原始資料
print("Data", response.data)

# 把回傳的 JSON 轉成 Python dict 再存取
data = response.get_json()

print(data["trails"])

Status 200
Data b'{\n    "message": "\\u6210\\u529f\\u67e5\\u5230\\u6240\\u6709\\u6b65\\u9053\\u57fa\\u672c\\u8cc7\\u6599",\n    "trails": [\n        {\n            "id": 1,\n            "name": "\\u6843\\u5c71\\u6b65\\u9053",\n            "location": "\\u81fa\\u4e2d\\u5e02\\u548c\\u5e73\\u5340,\\u65b0\\u7af9\\u7e23\\u5c16\\u77f3\\u9109",\n            "difficulty": "-",\n            "permitRequired": true\n        },\n        {\n            "id": 2,\n            "name": "\\u6843\\u5c71\\u5580\\u62c9\\u696d",\n            "location": "\\u81fa\\u4e2d\\u5e02\\u548c\\u5e73\\u5340,\\u65b0\\u7af9\\u7e23\\u5c16\\u77f3\\u9109,\\u5b9c\\u862d\\u7e23\\u5927\\u540c\\u9109",\n            "difficulty": "-",\n            "permitRequired": true\n        },\n        {\n            "id": 3,\n            "name": "\\u6c60\\u6709\\u5c71",\n            "location": "\\u81fa\\u4e2d\\u5e02\\u548c\\u5e73\\u5340,\\u65b0\\u7af9\\u7e23\\u5c16\\u77f3\\u9109",\n            "difficulty": "-",\n            "permitRequ

In [24]:
# Test: /api/trails/<id>

response = client.get("/api/trails/1")

print("Status", response.status_code)
# response.data 是 bytes 原始資料
print("Data", response.data)

# 把回傳的 JSON 轉成 Python dict 再存取
data = response.get_json()

print(response.status_code)

Status 500
Data b'{\n    "message": "\\u4f3a\\u670d\\u5668\\u932f\\u8aa4",\n    "error": "the JSON object must be str, bytes or bytearray, not list"\n}\n'
500


In [ ]:
from sqlalchemy import text

id = 1
with engine.connect() as conn:
    # 路線與氣象站詳細資料
    query_sql = """
                SELECT 
                    t.id as trail_id,
                    t.trail_name_ch,
                    t.location_name,
                    t.permit_required,
                    t.length_km,
                    t.elevation_start_m,
                    t.elevation_end_m,
                    json_agg(jsonb_build_object(
                        'station_id', s.id,
                        'station_code', s.station_code,
                        'station_name', s.station_name,
                        'station_geolocation', s.geolocation
                    )) AS stations
                FROM paths.trails t
                LEFT JOIN (
                    SELECT DISTINCT ON (ts.trail_id) *
                    FROM paths.trail_stations ts
                    ORDER BY ts.trail_id, ts.priority DESC
                ) ts ON t.id = ts.trail_id
                LEFT JOIN weather.stations s ON ts.station_id = s.id
                WHERE t.id = :trail_id
                GROUP BY t.id;
        """
    trail = conn.execute(text(query_sql), {"trail_id": id}).first()
    if not trail:
        print("找不到該步道")

    print(trail.stations[0])

    # stations =json.loads(trail.stations[0]) if trail.stations[0] else []

{'station_id': 4, 'station_code': 'C0F9Z0', 'station_name': '雪山東峰', 'station_geolocation': {'crs': {'type': 'name', 'properties': {'name': 'EPSG:4326'}}, 'type': 'Point', 'coordinates': [121.306, 24.403]}}


TypeError: the JSON object must be str, bytes or bytearray, not dict